In [ ]:
import subprocess, sys
pkgs = [
    "transformers==4.45.1", "bitsandbytes>=0.46.1", "peft>=0.4.0",
    "einops", "sentencepiece", "tiktoken", "decord", "av",
    "imageio", "opencv-python", "transformers_stream_generator",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✓ deps ready")

In [ ]:
import gc, glob, json, os, random, re, warnings
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image
from peft import LoraConfig, inject_adapter_in_model
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_PATH         = "OpenGVLab/InternVideo2_5_Chat_8B"
CHECKPOINT_PATH    = "/kaggle/working/qlora_checkpoints/best_checkpoint"
SAMPLED_FRAMES_DIR = "/kaggle/input/datasets/shresthml/sample-frame/sampled_frames"
SAMPLE_SUB_PATH    = "/kaggle/input/autopilot-reboot/sample_submission.csv"  # adjust if needed
OUTPUT_CSV         = "/kaggle/working/final_submission.csv"

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE_MAP = "auto"
MAX_MEMORY = {0: "13GiB", 1: "14GiB", "cpu": "24GiB"}

# ── Inference params ───────────────────────────────────────────────────────
NUM_FRAMES       = 20
FRAME_INPUT_SIZE = 224
MAX_NEW_TOKENS   = 512   # enough for the JSON response
IMAGENET_MEAN    = (0.485, 0.456, 0.406)
IMAGENET_STD     = (0.229, 0.224, 0.225)

# ── LoRA config (must match training) ─────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LLM_TARGET_MODULES = ["wqkv", "wo", "w1", "w2", "w3"]
VIT_TARGET_MODULES = []

print("✓ config ready")

In [ ]:
# ── All 25 questions: key → (description, {code: label}) ──────────────────
ALL_QUESTIONS = [
    ("Q1a", "Time of day",
     {0: "Day", 1: "Dawn (sunrise)", 2: "Dusk (sunset)", 3: "Night", -1: "Unknown"}),
    ("Q1b", "Weather conditions",
     {0: "Sunny/Clear", 1: "Cloudy", 2: "Rainy", 3: "Partly Cloudy",
      4: "Snowy", 5: "Foggy", -1: "Unknown"}),
    ("Q2",  "Lighting condition",
     {0: "Daylight", 1: "Headlights only",
      2: "Sunrise/Sunset – Early morning low-light, sun beginning to rise",
      4: "Streetlights on", -1: "Unknown"}),
    ("Q3a", "Traffic environment",
     {0: "Urban area – City streets with dense buildings, intersections, and traffic",
      2: "Suburban area – Residential or mixed-use commercial/residential area",
      3: "Rural area – Open or undeveloped roadside with sparse traffic",
      -1: "Unknown"}),
    ("Q3b", "Surrounding facilities",
     {1: "Urban area - skyscrapers, businesses, etc.",
      2: "Construction zone / Work area – Road partially blocked or altered due to work",
      3: "Residential Area", 4: "Parking lot / Gas station",
      5: "Near a school/college – Educational zone with potential pedestrian activity",
      -1: "Unknown"}),
    ("Q4a", "Road configuration",
     {1: "Highway", 2: "Intersection", 3: "T-junction / Y-junction",
      4: "Residential street", 5: "Bridge / Overpass / Underpass",
      6: "Roundabout / Traffic circle", 7: "Exit/Entrance ramp", -1: "Unknown"}),
    ("Q4b", "Lane configuration",
     {0: "Two-way divided", 1: "Two-way not divided", 2: "One-way", -1: "Unknown"}),
    ("Q4c", "Ego-car lane direction",
     {0: "Right", 1: "Left", 2: "Middle", -1: "Unknown"}),
    ("Q5a", "Incident category",
     {0: "Pedestrian/Cyclist/Vulnerable Road Users",
      1: "Vehicle-to-Vehicle", 2: "Animals / Objects", 3: "No Collision",
      5: "Barriers / Fixed Objects", 6: "Vehicle Loss of Control / Rollover",
      -1: "Unknown"}),
    ("Q5b", "Primary Incident Entity",
     {0: "Pedestrian", 1: "Ego-car", 2: "Animal", 3: "No collision",
      5: "Another Vehicle", 6: "Cyclist", 10: "Flying Object",
      13: "Smoke", 29: "Object on the road", -1: "Unknown"}),
    ("Q5c", "Primary entity vehicle type",
     {0: "Small Car", 1: "No collision", 2: "Truck", 3: "Bicycle",
      4: "Bicycle Cart", 5: "Motorcycle", 6: "Scooter",
      8: "SUV", 9: "Bus", -2: "Not applicable", -1: "Unknown"}),
    ("Q5e", "Primary entity behavior",
     {0: "Crossing", 1: "Speeding", 3: "Driving normally",
      4: "Overtaking/Changing Lanes", 6: "Turning", 7: "Walking",
      8: "Stopped", 10: "Swerving", 11: "Ignoring traffic signal",
      12: "Lost control", 13: "Fell Down", 14: "Rolling through",
      15: "Failure to yield", 16: "Flying", 17: "Rolling over",
      18: "Stationary", 19: "Fallen into the road", -1: "Unknown"}),
    ("Q5f", "Secondary Incident Entity",
     {0: "Cyclist", 1: "Ego-car", 2: "Another Vehicle",
      3: "Object or Barrier", 4: "Pedestrian", 5: "Scooter",
      6: "Animal", -1: "Unknown"}),
    ("Q5g", "Secondary entity vehicle type",
     {0: "Scooter", 1: "Small Car", 2: "Bicycle", 3: "Truck",
      4: "SUV", 5: "Bus", -2: "Not applicable", -1: "Unknown"}),
    ("Q5i", "Secondary entity behavior",
     {0: "Driving normally", 1: "Swerving", 2: "Fixed Position",
      3: "Speeding", 4: "Crossing", 5: "Turning", 6: "Lost control",
      7: "Failure to yield", 8: "Flying", 9: "Stopped",
      10: "Fallen onto road", 11: "Overtaking/Changing Lanes",
      13: "Rolling over", 14: "UTurn", -1: "Unknown"}),
    ("Q5j", "Incident peak",
     {0: "Near-miss", 1: "No collision", 2: "Collision",
      3: "No effect", 4: "Avoided in advance",
      5: "Chain-reaction", -1: "Unknown"}),
    ("Q6a", "Ego-car action",
     {0: "Going straight", 1: "Turning left", 2: "Turning right",
      3: "Reversing", 4: "Parked / Stationary", 5: "Changing lanes",
      6: "U-turn", -1: "Unknown"}),
    ("Q6b", "Ego-car speed",
     {0: "Stationary", 1: "Slow", 2: "Normal", 3: "Fast", -1: "Unknown"}),
    ("Q7a", "Traffic control",
     {0: "No control", 1: "Traffic lights", 2: "Stop sign",
      3: "Yield sign", 4: "Police officer", 5: "Other", -1: "Unknown"}),
    ("Q7b", "Traffic control status",
     {0: "Not applicable", 1: "Complied", 2: "Violated", -1: "Unknown"}),
    ("Q8",  "Number of vehicles involved",
     {0: "0", 1: "1", 2: "2", 3: "3+", -1: "Unknown"}),
    ("Q9a", "Road surface condition",
     {0: "Dry pavement", 1: "Wet road", 2: "Snow / Ice covered",
      3: "Debris / Mud on road", -1: "Unknown"}),
    ("Q9b", "Road material",
     {0: "Asphalt / Blacktop", 1: "Concrete",
      3: "Gravel / Dirt", -1: "Unknown"}),
    ("Q10a", "Visibility",
     {0: "Clear", 1: "Limited", 2: "Very poor", -1: "Unknown"}),
    ("Q10b", "Obstruction",
     {0: "None", 1: "Parked vehicles", 2: "Vegetation",
      3: "Buildings", 4: "Other", -1: "Unknown"}),
]

Q_KEYS   = [q[0] for q in ALL_QUESTIONS]
Q_DEFS   = {q[0]: (q[1], q[2]) for q in ALL_QUESTIONS}
# label → code (for submission encoding)
LBL2CODE = {k: {v: c for c, v in codes.items()} for k, (_, codes) in Q_DEFS.items()}
# default code per question (most conservative: -1 or first key)
DEFAULT  = {k: -1 for k in Q_KEYS}

print(f"✓ {len(Q_KEYS)} questions defined")

In [ ]:
import torch.nn as nn
from peft import LoraConfig, inject_adapter_in_model

def patch_internvl_source():
    import glob, os
    cache_root = os.path.expanduser("~/.cache/huggingface/hub")
    files = glob.glob(os.path.join(cache_root, "**", "modeling_internvl_chat*.py"), recursive=True)
    TARGET  = "input_embeds[selected] = vit_embeds"
    PATCHED = "input_embeds[selected] = vit_embeds.to(input_embeds.device)"
    for fpath in files:
        src = open(fpath).read()
        if PATCHED in src:
            print(f"  ✓ Already patched: {os.path.basename(fpath)}"); continue
        if TARGET not in src:
            continue
        open(fpath, "w").write(src.replace(TARGET, PATCHED))
        print(f"  ✓ Patched: {fpath}")


def _find_modules(model, wanted):
    found = set()
    for name, module in model.named_modules():
        for t in wanted:
            if name.endswith(t) and (isinstance(module, nn.Linear) or "Linear" in type(module).__name__):
                found.add(t)
    return list(found)


def load_inference_model():
    print("  Pre-fetching source files…")
    AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
    patch_internvl_source()

    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
    )
    model = AutoModel.from_pretrained(
        MODEL_PATH, trust_remote_code=True,
        device_map=DEVICE_MAP, max_memory=MAX_MEMORY,
        quantization_config=quant_cfg,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

    # ── Inject LoRA structure ────────────────────────────────────────────
    llm_found = _find_modules(model.language_model, LLM_TARGET_MODULES)
    vit_found = _find_modules(model.vision_model,   VIT_TARGET_MODULES)
    lora_cfg  = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", target_modules=list(set(llm_found + vit_found)),
    )
    model = inject_adapter_in_model(lora_cfg, model)

    # ── Load fine-tuned weights ──────────────────────────────────────────
    weights_path = os.path.join(CHECKPOINT_PATH, "lora_weights.pt")
    lora_cfg_path = os.path.join(CHECKPOINT_PATH, "lora_config.json")
    lora_state   = torch.load(weights_path, map_location="cpu")

    # Route each tensor to the device of its matching base layer
    name_to_device = {}
    for n, p in model.named_parameters():
        if "lora_" not in n and p.device.type != "cpu":
            # strip the param name to get the module prefix
            prefix = n.rsplit(".", 1)[0]
            name_to_device[prefix] = p.device

    loaded_state = {}
    for k, v in lora_state.items():
        prefix = k.rsplit(".", 1)[0]
        dev = name_to_device.get(prefix, torch.device("cuda:0"))
        loaded_state[k] = v.to(dev, dtype=torch.float16)

    missing, unexpected = model.load_state_dict(loaded_state, strict=False)
    print(f"  LoRA weights loaded — missing: {len(missing)}  unexpected: {len(unexpected)}")

    # Cast any float32 stragglers
    for _, p in model.named_parameters():
        if p.dtype == torch.float32:
            p.data = p.data.to(torch.float16)

    model.eval()

    # NUM_IMAGE_TOKEN
    num_image_token = getattr(model, "num_image_token", 16)
    img_ctx_id      = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
    model.img_context_token_id = img_ctx_id

    print(f"  NUM_IMAGE_TOKEN : {num_image_token}")
    print(f"  IMG_CONTEXT id  : {img_ctx_id}")
    print("  ✓ Model ready for inference")
    return model, tokenizer, num_image_token, img_ctx_id


model, tokenizer, NUM_IMAGE_TOKEN, IMG_CTX_ID = load_inference_model()

In [ ]:
def make_transform():
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((FRAME_INPUT_SIZE, FRAME_INPUT_SIZE), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

_TFM = make_transform()

def load_frames(frames_dir: str, n_frames: int = NUM_FRAMES):
    jpegs = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    if not jpegs:
        jpegs = sorted(glob.glob(os.path.join(frames_dir, "*.jpeg")))
    if not jpegs:
        raise FileNotFoundError(f"No frames in {frames_dir}")
    if len(jpegs) > n_frames:
        idx   = np.linspace(0, len(jpegs) - 1, n_frames, dtype=int)
        jpegs = [jpegs[i] for i in idx]
    return torch.stack([_TFM(Image.open(p).convert("RGB")) for p in jpegs])

print("✓ frame utils ready")

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert traffic accident analyst reviewing dashcam footage. "
    "The ego-car is the vehicle whose dashcam recorded the video."
)

def build_inference_prompt(num_frames: int) -> str:
    """Single prompt asking for all 25 questions at once."""
    frame_lines = "\n".join(f"Frame{i+1}: <image>" for i in range(num_frames))
    q_lines = []
    for key in Q_KEYS:
        desc, codes = Q_DEFS[key]
        opts = " | ".join(codes.values())
        q_lines.append(f"[{key}] {desc} — Options: {opts}")

    return (
        f"{frame_lines}\n\n"
        "Carefully analyse these dashcam frames.\n"
        "The ego-car is the vehicle whose dashcam recorded this video.\n\n"
        "Answer ALL of the following questions about this incident.\n"
        "Respond with ONLY a valid JSON object — no markdown, no explanation.\n\n"
        + "\n".join(q_lines)
        + "\n\nAnswer format:\n{\n"
        + ",\n".join(f'  "{k}": "<exact option text>"' for k in Q_KEYS)
        + "\n}"
    )


def build_input_ids(tokenizer, user_prompt: str, num_frames: int) -> dict:
    """Tokenise prompt with <IMG_CONTEXT> expansion, return model inputs."""
    img_ctx_str = tokenizer.decode([IMG_CTX_ID], skip_special_tokens=False)
    expanded    = user_prompt.replace("<image>", img_ctx_str * NUM_IMAGE_TOKEN)

    full_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{expanded}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    ids = tokenizer(full_text, add_special_tokens=False, return_tensors="pt").input_ids
    return ids


print("✓ prompt builder ready")

In [ ]:
def parse_json_response(text: str) -> dict:
    """Extract JSON from model output, tolerating markdown fences."""
    # strip markdown fences if present
    text = re.sub(r"```(?:json)?", "", text).strip()
    # find the first { ... } block
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        return {}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        # try to fix trailing commas
        fixed = re.sub(r",\s*([}\]])", r"\1", match.group())
        try:
            return json.loads(fixed)
        except Exception:
            return {}


def label_to_code(q_key: str, label: str) -> int:
    """Convert a predicted label string back to its numeric competition code."""
    mapping = LBL2CODE.get(q_key, {})
    # exact match
    if label in mapping:
        return mapping[label]
    # case-insensitive match
    label_l = label.strip().lower()
    for lbl, code in mapping.items():
        if lbl.lower() == label_l:
            return code
    # partial match (model sometimes truncates option text)
    for lbl, code in mapping.items():
        if label_l in lbl.lower() or lbl.lower() in label_l:
            return code
    return -1   # Unknown fallback


print("✓ parser ready")

In [ ]:
def get_vit_device(model):
    try:    return next(model.vision_model.parameters()).device
    except: return torch.device("cuda:0")

def get_llm_device(model):
    try:
        emb = (getattr(model.language_model.model, "tok_embeddings", None)
               or getattr(model.language_model.model, "embed_tokens", None))
        if emb: return next(emb.parameters()).device
    except: pass
    return torch.device("cuda:0")


def run_inference(model, tokenizer, video_id: int) -> dict:
    """Run inference for one video, return {Q_key: numeric_code}."""
    fdir = os.path.join(SAMPLED_FRAMES_DIR, str(video_id))

    try:
        pixel_values = load_frames(fdir, NUM_FRAMES)
    except FileNotFoundError:
        print(f"  WARNING: no frames for video {video_id} — using defaults")
        return dict(DEFAULT)

    n_frames   = pixel_values.shape[0]
    vit_dev    = get_vit_device(model)
    llm_dev    = get_llm_device(model)

    user_prompt = build_inference_prompt(n_frames)
    input_ids   = build_input_ids(tokenizer, user_prompt, n_frames).to(llm_dev)
    pixel_values = pixel_values.to(vit_dev, dtype=torch.float16, non_blocking=True)
    image_flags  = torch.ones(n_frames, dtype=torch.long).to(llm_dev)

    with torch.no_grad():
        try:
            output_ids = model.generate(
                input_ids    = input_ids,
                pixel_values = pixel_values,
                image_flags  = image_flags,
                max_new_tokens = MAX_NEW_TOKENS,
                do_sample      = False,      # greedy — deterministic, fastest
                use_cache      = True,
            )
        except Exception as e:
            print(f"  ERROR video {video_id}: {e}")
            return dict(DEFAULT)

    # Decode only the new tokens
    new_tokens = output_ids[0, input_ids.shape[1]:]
    raw_text   = tokenizer.decode(new_tokens, skip_special_tokens=True)

    parsed = parse_json_response(raw_text)

    result = {}
    for key in Q_KEYS:
        label       = parsed.get(key, "Unknown")
        result[key] = label_to_code(key, label)

    return result


# ── Run over all 661 videos ────────────────────────────────────────────────
# Load video IDs from sample_submission if available, else 0-660
try:
    sample_df  = pd.read_csv(SAMPLE_SUB_PATH)
    video_ids  = sample_df.iloc[:, 0].astype(int).tolist()
    sub_cols   = list(sample_df.columns)
    print(f"  Loaded {len(video_ids)} video IDs from sample_submission")
except Exception:
    video_ids = list(range(661))
    sub_cols  = None
    print("  sample_submission not found — using video IDs 0-660")

records = []
for vid_id in tqdm(video_ids, desc="Inference"):
    row = run_inference(model, tokenizer, vid_id)
    row["video_id"] = vid_id
    records.append(row)
    # light cleanup every 50 videos
    if vid_id % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n✓ Inference complete — {len(records)} videos processed")

In [ ]:
# ── Build submission DataFrame ─────────────────────────────────────────────
results_df = pd.DataFrame(records)
results_df = results_df.set_index("video_id")

# If sample_submission exists, use its exact column order and fill missing cols
if sub_cols is not None:
    vid_col   = sub_cols[0]
    ans_cols  = [c for c in sub_cols if c != vid_col]
    # Map Q_KEYS to submission column names via sample_submission header
    # (competition uses verbose column names — keep them from sample_submission)
    sub_df = sample_df.copy()
    sub_df = sub_df.set_index(vid_col)

    # Best-effort: match Q-key to sample column by finding Q-key substring
    col_to_key = {}
    for col in ans_cols:
        for key in Q_KEYS:
            if key in col:
                col_to_key[col] = key
                break

    for col, key in col_to_key.items():
        sub_df[col] = sub_df.index.map(
            lambda vid, k=key: results_df.loc[vid, k] if vid in results_df.index else -1
        )

    sub_df = sub_df.reset_index()
    sub_df.to_csv(OUTPUT_CSV, index=False)
else:
    # Simple format: video_id + Q1a ... Q10b
    out = results_df.reset_index()[["video_id"] + Q_KEYS]
    out.to_csv(OUTPUT_CSV, index=False)

print(f"✓ Saved: {OUTPUT_CSV}")
print(f"  Shape : {pd.read_csv(OUTPUT_CSV).shape}")
pd.read_csv(OUTPUT_CSV).head(3)